This notebook aims to build a product recommendation system using an embedding model based on **Sentence Transformers**.  
The goal is to semantically understand a user's query and recommend the most relevant products from the store.

We will use the **Flipkart E-commerce dataset** from Hugging Face.    
This dataset contains product information like names, descriptions, specifications, prices, and ratings.

After exploring the dataset, we will:
- Combine key textual features such as **product name**, **description**, and **specifications** into a single column  
- Use this combined text to create **semantic embeddings**  
- These embeddings will allow us to compare user queries with products and return the most relevant recommendations

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from zipfile import ZipFile
import re

In [3]:
os.getcwd()

'/content'

In [4]:
working_directory = "/content/drive/MyDrive/transformers/Ecommerce/ProductRecommendations"

In [5]:
os.chdir(working_directory)

In [6]:
os.getcwd()

'/content/drive/MyDrive/transformers/Ecommerce/ProductRecommendations'

In [7]:
! ls

FlipKart_ecommerce.zip	product_recommendation.ipynb


In [8]:
with ZipFile("./FlipKart_ecommerce.zip", "r") as f:
  f.extractall()

In [9]:
! ls

flipkart_com-ecommerce_sample.csv  product_recommendation.ipynb
FlipKart_ecommerce.zip


In [10]:
csv_file_path = "./flipkart_com-ecommerce_sample.csv"

In [11]:
df = pd.read_csv(csv_file_path)

In [12]:
df.head()

,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_FK_Advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2FF9KEDEFGF,999.0,379.0,"[""http://img5a.flixcart.com/image/short/u/4/a/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
1,7f7036a6d550aaa89d34c77bd39a5e48,2016-03-25 22:59:23 +0000,http://www.flipkart.com/fabhomedecor-fabric-do...,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",SBEEH3QGU7MFYJFY,32157.0,22646.0,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",False,FabHomeDecor Fabric Double Sofa Bed (Finish Co...,No rating available,No rating available,FabHomeDecor,"{""product_specification""=>[{""key""=>""Installati..."
2,f449ec65dcbc041b6ae5e6a32717d01b,2016-03-25 22:59:23 +0000,http://www.flipkart.com/aw-bellies/p/itmeh4grg...,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",SHOEH4GRSUBJGZXE,999.0,499.0,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",False,Key Features of AW Bellies Sandals Wedges Heel...,No rating available,No rating available,AW,"{""product_specification""=>[{""key""=>""Ideal For""..."
3,0973b37acd0c664e3de26e97e5571454,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2F6HUZMQ6SJ,699.0,267.0,"[""http://img5a.flixcart.com/image/short/6/2/h/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
4,bc940ea42ee6bef5ac7cea3fb5cfbee7,2016-03-25 22:59:23 +0000,http://www.flipkart.com/sicons-all-purpose-arn...,Sicons All Purpose Arnica Dog Shampoo,"[""Pet Supplies >> Grooming >> Skin & Coat Care...",PSOEH3ZYDMSYARJ5,220.0,210.0,"[""http://img5a.flixcart.com/image/pet-shampoo/...",False,Specifications of Sicons All Purpose Arnica Do...,No rating available,No rating available,Sicons,"{""product_specification""=>[{""key""=>""Pet Type"",..."


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20002 entries, 0 to 20001
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   uniq_id                  20000 non-null  object 
 1   crawl_timestamp          20000 non-null  object 
 2   product_url              20000 non-null  object 
 3   product_name             20000 non-null  object 
 4   product_category_tree    20000 non-null  object 
 5   pid                      20000 non-null  object 
 6   retail_price             19922 non-null  float64
 7   discounted_price         19922 non-null  float64
 8   image                    19997 non-null  object 
 9   is_FK_Advantage_product  20000 non-null  object 
 10  description              19998 non-null  object 
 11  product_rating           20000 non-null  object 
 12  overall_rating           20000 non-null  object 
 13  brand                    14136 non-null  object 
 14  product_specifications

In [14]:
df["description"] = df["description"].fillna("No Product Description")
df["product_specifications"] = df["product_specifications"].fillna("No Product Specification")
df["product_name"] = df["product_name"].fillna("No Product Name")

In [15]:
df["product_name"][0]

"Alisha Solid Women's Cycling Shorts"

In [16]:
df["description"][0]

"Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra Navy, Red, Navy,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 3 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women's Fabric Care Gentle Machine Wash in Lukewarm Water, Do Not Bleach Additional Details Style Code ALTHT_3P_21 In the Box 3 shorts"

In [17]:
df["product_specifications"][0]

'{"product_specification"=>[{"key"=>"Number of Contents in Sales Package", "value"=>"Pack of 3"}, {"key"=>"Fabric", "value"=>"Cotton Lycra"}, {"key"=>"Type", "value"=>"Cycling Shorts"}, {"key"=>"Pattern", "value"=>"Solid"}, {"key"=>"Ideal For", "value"=>"Women\'s"}, {"value"=>"Gentle Machine Wash in Lukewarm Water, Do Not Bleach"}, {"key"=>"Style Code", "value"=>"ALTHT_3P_21"}, {"value"=>"3 shorts"}]}'

In [18]:
sentence = df["product_specifications"][0]

In [19]:
pattern = r'{"key"=>"(.*?)", "value"=>"(.*?)"}'

In [20]:
pairs = re.findall(pattern, sentence)

In [21]:
spec_text = "Product specifications: " + "; ".join(
    f"{key.strip()}: {value.strip()}"
    for key, value in pairs
    if key and value
)

In [22]:
spec_text

"Product specifications: Number of Contents in Sales Package: Pack of 3; Fabric: Cotton Lycra; Type: Cycling Shorts; Pattern: Solid; Ideal For: Women's; Style Code: ALTHT_3P_21"

### Create a function to extarct specifications of each product

In [23]:
def extract_specifications(specifications):
  pattern = r'{"key"=>"(.*?)", "value"=>"(.*?)"}'
  pairs = re.findall(pattern, sentence)
  spec_text = "Product specifications: " + "; ".join(
    f"{key.strip()}: {value.strip()}"
    for key, value in pairs
    if key and value
)
  return spec_text

In [24]:
df["specifications"] = df['product_specifications'].apply(extract_specifications)

In [25]:
df["specifications"] = df["specifications"].fillna("No Product specifications")

In [26]:
df["product_description_full"] = "Product Name: " + df["product_name"].str.strip() + "\n" + "Product Description : " +  df["description"].str.strip() + "\n" + df["specifications"]

In [27]:
print(df["product_description_full"][15])

Product Name: Alisha Solid Women's Cycling Shorts
Product Description : Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra Black, White, White,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 3 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women's Fabric Care Gentle Machine Wash in Lukewarm Water, Do Not Bleach Additional Details Style Code ALTHT_3P_2 In the Box 3 shorts
Product specifications: Number of Contents in Sales Package: Pack of 3; Fabric: Cotton Lycra; Type: Cycling Shorts; Pattern: Solid; Ideal For: Women's; Style Code: ALTHT_3P_21


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20002 entries, 0 to 20001
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   uniq_id                   20000 non-null  object 
 1   crawl_timestamp           20000 non-null  object 
 2   product_url               20000 non-null  object 
 3   product_name              20002 non-null  object 
 4   product_category_tree     20000 non-null  object 
 5   pid                       20000 non-null  object 
 6   retail_price              19922 non-null  float64
 7   discounted_price          19922 non-null  float64
 8   image                     19997 non-null  object 
 9   is_FK_Advantage_product   20000 non-null  object 
 10  description               20002 non-null  object 
 11  product_rating            20000 non-null  object 
 12  overall_rating            20000 non-null  object 
 13  brand                     14136 non-null  object 
 14  produc

### Embedding Text Using Sentence Transformers

In [29]:
model_name = "all-mpnet-base-v2"

In [ ]:
model = SentenceTransformer(model_name)

In [31]:
sentence_embeddings = model.encode(df["product_description_full"].tolist())

In [32]:
type(sentence_embeddings)

numpy.ndarray

In [33]:
sentence_embeddings.shape

(20002, 768)

In [34]:
working_directory = "/content/drive/MyDrive/transformers/Ecommerce/ProductRecommendations"

In [35]:
embeddings_path = working_directory + "/embeddings.npy"

In [36]:
np.save(embeddings_path, sentence_embeddings)

In [37]:
! ls

embeddings.npy			   FlipKart_ecommerce.zip
flipkart_com-ecommerce_sample.csv  product_recommendation.ipynb
